In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, upper
from pyspark.sql.types import StringType, StructField, StructType

from pipelines.bronze.stream_utils import (
    read_autoloader_stream,
    read_delta_table_stream,
    write_delta_table_stream,
)

In [0]:
source_path = "/Volumes/main/bronze/ecommerce/clicks"
source_schema_location = "/Volumes/main/bronze/ecommerce/_schemas/clicks"

target_path = "/Volumes/main/silver/ecommerce/clicks"
target_schema_location = "/Volumes/main/silver/ecommerce/_schemas/clicks"
checkpoint_location = "/Volumes/main/silver/ecommerce/_checkpoints/clicks"

In [0]:
source_stream_df = read_autoloader_stream(
    source_path=source_path,
    schema_location=source_schema_location,
    schema_evolution_mode="addNewColumns"
)

In [0]:
silver_stream_df = (
    source_stream_df
    .withColumn("occurred_at", to_timestamp(col("occurred_at")))
    .withColumn("event_date", to_date(col("occurred_at")))
)

In [0]:
silver_query = write_delta_table_stream(
    df=silver_stream_df,
    table_name="main.silver.clicks",
    checkpoint_location=checkpoint_location,
    query_name="silver_clicks_available_now",
    partition_by=["event_date"],
    merge_schema=True
)

silver_query.awaitTermination()